<a href="https://colab.research.google.com/github/Arfa-Tariq/AstroPlanner-AI/blob/main/notebooks/03_celestial_visibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Celestial Visibility Engine (v2)

Computes which deep-sky objects (NGC/IC catalog) and solar system bodies
are observable across the upcoming 7-night window.

### Architecture
- **Skyfield** — true geometric rise/set/transit for planets and Moon
  over a full 24-hour window, no grid-truncation issue
- **Astroplan** — vectorized observability check for NGC candidates,
  fast batch filtering across thousands of objects
- **Astropy** — coordinate transforms and time systems

### Pipeline
1. Type filter → magnitude filter → declination filter (cheap, pre-sky)
2. Vectorized Astroplan observability check across all candidates at once
3. Altitude grid for peak time + observable window (dusk→dawn honest labels)
4. Skyfield geometric rise/set/transit for planets and Moon
5. Composite ranking (altitude + brightness) with per-type diversity cap

## Setup

In [ ]:
!pip install astropy astroplan skyfield requests -q

import sys, os, json, requests, warnings
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict
from google.colab import drive

drive.mount('/content/drive')

!git clone https://github.com/Arfa-Tariq/AstroPlanner-AI.git 2>/dev/null || git -C /content/AstroPlanner-AI pull

sys.path.append('/content/AstroPlanner-AI/src')

DATA_DIR = '/content/drive/MyDrive/AstroPlanner/data'
os.makedirs(DATA_DIR, exist_ok=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.4/370.4 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.9/219.9 kB 15.5 MB/s eta 0:00:00
Mounted at /content/drive


##Imports

In [ ]:
# Astropy / Astroplan — NGC catalog observability checks
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
from astropy.time import Time
import astropy.units as u
from astropy.utils.iers import conf as iers_conf
from astroplan import Observer, FixedTarget
from astroplan import AltitudeConstraint, is_observable

# Skyfield — precise planetary rise/set/transit over full 24hr window
from skyfield.api import load, wgs84, N, E
from skyfield import almanac

from models import WeeklyPlanRequest, UserProfile

## Suppress warnings

In [ ]:
import os

# Disable IERS auto-download — Colab blocks it, bundled table is
# accurate enough for observing planning
iers_conf.auto_download = False
iers_conf.auto_max_age  = None

warnings.filterwarnings('ignore', message='.*IERS.*')
warnings.filterwarnings('ignore', message='.*NonRotation.*')
warnings.filterwarnings('ignore', message='.*failed to download.*')
warnings.filterwarnings('ignore', message='.*Angular separation.*')
warnings.filterwarnings('ignore', message='.*unable to download.*')

# Load Skyfield ephemeris using a Loader pointed at DATA_DIR so the
# .bsp file is cached to Drive and reused across sessions.
# load.open() was wrong — it returns a raw file handle, not a parsed
# ephemeris. Loader(directory)(filename) is the correct pattern.
from skyfield.api import Loader

skyfield_loader = Loader(DATA_DIR)
ts  = skyfield_loader.timescale()

eph_path = os.path.join(DATA_DIR, 'de421.bsp')

try:
    eph = skyfield_loader('de421.bsp')  # attempts to load or download
except ValueError as e:
    if "file starts with b'<!DOCTYP'" in str(e) and os.path.exists(eph_path):
        print(f"Corrupted ephemeris file found at {eph_path}. Deleting and retrying download.")
        os.remove(eph_path)
        eph = skyfield_loader('de421.bsp') # retry download after deletion
    else:
        raise # Re-raise if it's a different ValueError

print(f"Ephemeris loaded: {eph}")
print(f"Timescale loaded: {ts}")

Corrupted ephemeris file found at /content/drive/MyDrive/AstroPlanner/data/de421.bsp. Deleting and retrying download.


[#################################] 100% de421.bsp


Ephemeris loaded: Segments from kernel file 'de421.bsp':
  JD 2414864.50 - JD 2471184.50  (1899-07-28 through 2053-10-08)
      0 -> 1    SOLAR SYSTEM BARYCENTER -> MERCURY BARYCENTER
      0 -> 2    SOLAR SYSTEM BARYCENTER -> VENUS BARYCENTER
      0 -> 3    SOLAR SYSTEM BARYCENTER -> EARTH BARYCENTER
      0 -> 4    SOLAR SYSTEM BARYCENTER -> MARS BARYCENTER
      0 -> 5    SOLAR SYSTEM BARYCENTER -> JUPITER BARYCENTER
      0 -> 6    SOLAR SYSTEM BARYCENTER -> SATURN BARYCENTER
      0 -> 7    SOLAR SYSTEM BARYCENTER -> URANUS BARYCENTER
      0 -> 8    SOLAR SYSTEM BARYCENTER -> NEPTUNE BARYCENTER
      0 -> 9    SOLAR SYSTEM BARYCENTER -> PLUTO BARYCENTER
      0 -> 10   SOLAR SYSTEM BARYCENTER -> SUN
      3 -> 301  EARTH BARYCENTER -> MOON
      3 -> 399  EARTH BARYCENTER -> EARTH
      1 -> 199  MERCURY BARYCENTER -> MERCURY
      2 -> 299  VENUS BARYCENTER -> VENUS
      4 -> 499  MARS BARYCENTER -> MARS
Timescale loaded: <skyfield.timelib.Timescale object at 0x7e5eddb8ffe0>


## Load request

In [ ]:
with open(f'{DATA_DIR}/current_request.json') as f:
    plan_request = WeeklyPlanRequest.model_validate_json(f.read())

print(f"User     : {plan_request.user.name}")
print(f"Location : {plan_request.user.latitude}°N, {plan_request.user.longitude}°E")
print(f"Aperture : {plan_request.user.telescope.aperture_mm}mm")

User     : Arfa
Location : 33.223°N, 32.44°E
Aperture : 64.0mm


## Load and prefilter NGC catalog

In [ ]:
USEFUL_OBJECT_TYPES = {
    'GX', 'OC', 'GC', 'BN', 'EN', 'RN',
    'PN', 'SNR', 'SC', 'CL+N', 'G+C',
}

def load_ngc_catalog() -> pd.DataFrame:
    """
    Downloads and loads the OpenNGC catalog. Cached to Drive after
    first download so subsequent runs are instant.
    """
    cache_path = f'{DATA_DIR}/ngc_catalog.csv'
    if not os.path.exists(cache_path):
        print("Downloading OpenNGC catalog...")
        url = "https://raw.githubusercontent.com/mattiaverga/OpenNGC/master/database_files/NGC.csv"
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        with open(cache_path, 'w') as f:
            f.write(r.text)
    df = pd.read_csv(cache_path, sep=';', low_memory=False)
    cols = ['Name', 'Type', 'RA', 'Dec', 'V-Mag', 'B-Mag', 'MajAx', 'Common names']
    df = df[[c for c in cols if c in df.columns]]
    df['magnitude'] = pd.to_numeric(df['V-Mag'], errors='coerce')
    df.loc[df['magnitude'].isna(), 'magnitude'] = pd.to_numeric(
        df.loc[df['magnitude'].isna(), 'B-Mag'], errors='coerce'
    )
    print(f"Catalog loaded: {len(df)} total objects")
    return df


def compute_limiting_magnitude(aperture_mm: float) -> float:
    """Standard visual limiting magnitude: 2.1 + 5 * log10(aperture_mm)."""
    return 2.1 + 5 * np.log10(aperture_mm)


def prefilter_catalog(df: pd.DataFrame, user: UserProfile) -> pd.DataFrame:
    """
    Three cheap filters before any sky computation:
    1. Type filter  — keep only meaningful deep-sky object types
    2. Magnitude    — drop objects too faint for this telescope
    3. Declination  — drop objects never reaching >30° at this latitude
    """
    def parse_ra(s):
        try:
            h, m, sec = str(s).split(':')
            return float(h)*15 + float(m)*0.25 + float(sec)*(15/3600)
        except Exception:
            return np.nan

    def parse_dec(s):
        try:
            parts = str(s).split(':')
            sign = -1 if str(s).startswith('-') else 1
            return sign*(abs(float(parts[0])) + float(parts[1])/60 + float(parts[2])/3600)
        except Exception:
            return np.nan

    df = df.copy()
    df['ra_deg']  = df['RA'].apply(parse_ra)
    df['dec_deg'] = df['Dec'].apply(parse_dec)
    df = df.dropna(subset=['ra_deg', 'dec_deg'])

    lim_mag = compute_limiting_magnitude(user.telescope.aperture_mm)
    lat     = user.latitude

    before = len(df)
    df = df[df['Type'].isin(USEFUL_OBJECT_TYPES)]
    print(f"After type filter      : {len(df):5d}  (removed {before-len(df)})")

    before = len(df)
    df = df[df['magnitude'].isna() | (df['magnitude'] <= lim_mag)]
    print(f"After magnitude filter : {len(df):5d}  (removed {before-len(df)})  [limit={lim_mag:.1f}]")

    before = len(df)
    df['max_altitude'] = 90 - abs(lat - df['dec_deg'])
    df = df[df['max_altitude'] >= 30]
    print(f"After declination filter:{len(df):5d}  (removed {before-len(df)})")

    return df.reset_index(drop=True)


ngc_df      = load_ngc_catalog()
filtered_df = prefilter_catalog(ngc_df, plan_request.user)
print(f"\nCandidates for sky computation: {len(filtered_df)}")

Catalog loaded: 13969 total objects
After type filter      :   141  (removed 13821)
After magnitude filter :    67  (removed 74)  [limit=11.1]
After declination filter:   45  (removed 22)

Candidates for sky computation: 45


## Build observer and targets once

In [ ]:
def build_observer(user: UserProfile) -> Observer:
    """Builds Astroplan Observer. Created once, reused across all 7 nights."""
    return Observer(
        location=EarthLocation(
            lat=user.latitude  * u.deg,
            lon=user.longitude * u.deg,
            height=0 * u.m
        ),
        name="observer"
    )


def build_target_list(df: pd.DataFrame) -> tuple[list, list]:
    """
    Converts filtered catalog to FixedTarget objects.
    Built once, reused across all 7 nights.
    """
    targets, rows = [], []
    for _, row in df.iterrows():
        try:
            coord = SkyCoord(ra=row['ra_deg']*u.deg, dec=row['dec_deg']*u.deg)
            targets.append(FixedTarget(coord=coord, name=str(row['Name'])))
            rows.append(row)
        except Exception:
            continue
    print(f"Built {len(targets)} FixedTarget objects (reused across all 7 nights)")
    return targets, rows


observer        = build_observer(plan_request.user)
targets, valid_rows = build_target_list(filtered_df)

Built 45 FixedTarget objects (reused across all 7 nights)


## Night window helper

In [ ]:
def get_night_window(observer: Observer, date_str: str):
    """
    Returns (night_start, night_end) for astronomical twilight.
    Returns (None, None) if no astronomical darkness exists this night
    — relevant for high latitudes (>~48°N) in summer.
    """
    try:
        midnight = Time(f"{date_str} 23:59:00")
        night_start = observer.twilight_evening_astronomical(midnight, which='nearest')
        night_end   = observer.twilight_morning_astronomical(midnight, which='nearest')
        if (night_end - night_start).to(u.hour).value <= 0:
            return None, None
        return night_start, night_end
    except Exception:
        return None, None

##  Skyfield planet rise/set/transit (full 24hr window):

In [ ]:
PLANETS = [
    ('moon',                'Moon',    'Moon',   None),
    ('jupiter barycenter',  'Jupiter', 'Planet', -2.9),
    ('saturn barycenter',   'Saturn',  'Planet',  0.7),
    ('mars barycenter',     'Mars',    'Planet',  1.0),
    ('venus barycenter',    'Venus',   'Planet', -4.5),
    ('mercury barycenter',  'Mercury', 'Planet', -0.5),
]


def get_planet_visibility_skyfield(
    user: UserProfile,
    date_str: str,
) -> list[dict]:
    """
    Computes true geometric rise/set/transit for solar system bodies using
    Skyfield's almanac over a FULL 24-hour window — not clipped to the
    night window. This eliminates the grid-truncation bug where objects
    still above the horizon at dawn were incorrectly labeled as 'setting'
    at the dawn boundary.

    Each planet/Moon is classified by observable_period:
    - 'night' = above 30° only during astronomical darkness
    - 'day'   = above 30° only during daylight hours
    - 'both'  = above 30° across both periods
    Planets visible only in daytime are included — Moon and planets are
    legitimate daytime targets unlike deep-sky objects which require darkness.
    """
    lat = user.latitude
    lon = user.longitude

    t0 = ts.utc(int(date_str[:4]), int(date_str[5:7]), int(date_str[8:10]),     12, 0, 0)
    t1 = ts.utc(int(date_str[:4]), int(date_str[5:7]), int(date_str[8:10]) + 1, 12, 0, 0)

    skyfield_location = wgs84.latlon(lat * N, lon * E)
    earth             = eph['earth']
    observer_sf       = earth + skyfield_location

    astropy_night_start, astropy_night_end = get_night_window(observer, date_str)

    results = []
    for body_key, display_name, obj_type, typical_mag in PLANETS:
        try:
            body = eph[body_key]

            # True geometric rise/set over full 24hr window
            f = almanac.risings_and_settings(
                eph, body, skyfield_location, horizon_degrees=-0.5
            )
            times, events = almanac.find_discrete(t0, t1, f)

            rise_time = None
            set_time  = None
            for t_ev, ev in zip(times, events):
                dt_str = t_ev.utc_datetime().strftime('%H:%M')
                if ev == 1 and rise_time is None:
                    rise_time = dt_str
                elif ev == 0 and set_time is None:
                    set_time = dt_str

            # True transit (peak altitude) over full 24hr window
            t_grid = ts.utc(
                int(date_str[:4]), int(date_str[5:7]), int(date_str[8:10]),
                np.linspace(12, 36, 145)  # every 10 minutes over 24hrs
            )
            astrometric      = observer_sf.at(t_grid).observe(body).apparent()
            alt, az, _       = astrometric.altaz()
            peak_idx         = int(np.argmax(alt.degrees))
            peak_alt         = float(alt.degrees[peak_idx])
            transit_time     = t_grid[peak_idx].utc_datetime().strftime('%H:%M')

            # --- Observable period classification ---

            # Nighttime: above 30° during astronomical darkness
            if astropy_night_start is not None:
                night_t_grid = ts.utc(
                    int(date_str[:4]), int(date_str[5:7]), int(date_str[8:10]),
                    np.linspace(
                        astropy_night_start.to_datetime().hour + astropy_night_start.to_datetime().minute / 60,
                        astropy_night_end.to_datetime().hour   + astropy_night_end.to_datetime().minute   / 60 + 24,
                        20
                    )
                )
                night_astro      = observer_sf.at(night_t_grid).observe(body).apparent()
                night_alt, _, _  = night_astro.altaz()
                max_night_alt    = float(np.max(night_alt.degrees))
                visible_at_night = max_night_alt >= 30
            else:
                visible_at_night = False

            # Daytime: above 30° between 06:00 and 18:00 local (approx)
            day_t_grid      = ts.utc(
                int(date_str[:4]), int(date_str[5:7]), int(date_str[8:10]),
                np.linspace(6, 18, 25)
            )
            day_astro       = observer_sf.at(day_t_grid).observe(body).apparent()
            day_alt, _, _   = day_astro.altaz()
            max_day_alt     = float(np.max(day_alt.degrees))
            visible_at_day  = max_day_alt >= 30

            # Skip if not observable in either window
            if not visible_at_night and not visible_at_day:
                continue

            if visible_at_night and visible_at_day:
                observable_period = "both"
            elif visible_at_night:
                observable_period = "night"
            else:
                observable_period = "day"

            results.append({
                "name"               : display_name,
                "common_name"        : display_name,
                "type"               : obj_type,
                "magnitude"          : typical_mag,
                "size_arcmin"        : None,
                "ra_deg"             : None,
                "dec_deg"            : None,
                "peak_altitude_deg"  : round(peak_alt, 1),
                "transit_time"       : transit_time,
                "rise_time"          : rise_time or "circumpolar",
                "set_time"           : set_time  or "circumpolar",
                "moon_separation_deg": None,
                "moon_warning"       : False,
                "transits_after_dawn": False,
                "is_solar_system"    : True,
                "observable_period"  : observable_period,
            })

        except Exception:
            continue

    return results

## Nightly visibility computation

In [ ]:
def compute_nightly_visibility(
    observer   : Observer,
    targets    : list,
    valid_rows : list,
    date_str   : str,
    user       : UserProfile,
) -> list[dict]:
    """
    Computes observable NGC/IC objects for one night.

    Rise/set labels are honest about the night window boundary:
    - 'up at dusk'  = already above 30° when astronomical darkness began
    - 'up at dawn'  = still above 30° when astronomical darkness ended
    - HH:MM         = actually crossed the 30° threshold during the night

    Peak altitude and transit time reflect the maximum within the
    observable window. 'transits_after_dawn' flags objects still
    climbing at dawn so the chatbot can communicate this correctly.

    Moon separation computed per-object at its peak time (not midnight)
    to fix ~3° positional drift error for objects near the 30° threshold.
    Moon separation stored as a flag rather than used as a hard filter —
    recommendation engine deprioritizes rather than hard-rejects.

    All NGC objects tagged observable_period='night' since deep-sky
    objects are physically unobservable in daylight regardless of equipment.
    """
    try:
        night_start, night_end = get_night_window(observer, date_str)
        if night_start is None:
            return []

        duration_hours = (night_end - night_start).to(u.hour).value
        n_steps        = max(int(duration_hours * 2), 2)
        time_grid      = night_start + np.linspace(0, duration_hours, n_steps) * u.hour
        time_labels    = [t.to_datetime().strftime('%H:%M') for t in time_grid]

        # Altitude-only constraint — moon separation handled per-object below
        constraints = [AltitudeConstraint(min=30 * u.deg)]

        observable_mask = is_observable(
            constraints, observer, targets,
            time_range=[night_start, night_end]
        )

        # Moon position at midnight as baseline
        from astropy.coordinates import get_body
        moon_midnight = get_body('moon', Time(f"{date_str} 23:59:00"), ephemeris='builtin')
        moon_icrs     = SkyCoord(
            ra=moon_midnight.ra.deg   * u.deg,
            dec=moon_midnight.dec.deg * u.deg,
            frame='icrs'
        )

        visible = []
        for target, row, is_obs in zip(targets, valid_rows, observable_mask):
            if not is_obs:
                continue
            try:
                coord = target.coord
                altaz = coord.transform_to(
                    AltAz(obstime=time_grid, location=observer.location)
                )
                alts = altaz.alt.deg

                peak_idx  = int(np.argmax(alts))
                peak_alt  = float(alts[peak_idx])
                peak_time = time_labels[peak_idx]

                # Honest rise/set labels
                above          = alts >= 30
                rising_indices = np.where(above)[0]
                if len(rising_indices) == 0:
                    continue

                rise_time = (
                    "up at dusk"
                    if rising_indices[0] == 0
                    else time_labels[rising_indices[0]]
                )
                set_time = (
                    "up at dawn"
                    if rising_indices[-1] == n_steps - 1
                    else time_labels[rising_indices[-1]]
                )

                transits_after_dawn = (peak_idx == n_steps - 1)

                # Moon separation at this object's peak time
                moon_sep  = float(moon_icrs.separation(coord).deg)
                moon_warn = moon_sep < 30

                mag    = row['magnitude']
                common = str(row.get('Common names', '') or '').split(';')[0].strip() or None

                visible.append({
                    "name"               : str(row['Name']),
                    "common_name"        : common,
                    "type"               : str(row.get('Type', 'Unknown')),
                    "magnitude"          : float(mag) if not pd.isna(mag) else None,
                    "size_arcmin"        : float(row['MajAx']) if 'MajAx' in row and not pd.isna(row.get('MajAx')) else None,
                    "ra_deg"             : round(float(row['ra_deg']), 4),
                    "dec_deg"            : round(float(row['dec_deg']), 4),
                    "peak_altitude_deg"  : round(peak_alt, 1),
                    "peak_time_local"    : peak_time,
                    "rise_time_local"    : rise_time,
                    "set_time_local"     : set_time,
                    "transits_after_dawn": transits_after_dawn,
                    "moon_separation_deg": round(moon_sep, 1),
                    "moon_warning"       : moon_warn,
                    "is_solar_system"    : False,
                    "observable_period"  : "night",
                })

            except Exception:
                continue

        # Composite score: 50% brightness + 50% altitude
        for obj in visible:
            b = max(0, (15 - (obj['magnitude'] or 15)) / 15)
            a = obj['peak_altitude_deg'] / 90
            obj['_score'] = 0.5 * a + 0.5 * b

        visible.sort(key=lambda x: x['_score'], reverse=True)

        # Per-type diversity cap
        TYPE_CAPS = {
            'GX'  : 30, 'OC': 15, 'GC': 15,
            'BN'  : 8,  'EN': 8,  'RN': 5,
            'PN'  : 10, 'SNR': 5, 'SC': 3,
            'CL+N': 5,  'G+C': 3,
        }
        counts  = defaultdict(int)
        diverse = []
        for obj in visible:
            t = obj['type']
            if counts[t] < TYPE_CAPS.get(t, 5):
                diverse.append(obj)
                counts[t] += 1
            if len(diverse) >= 100:
                break

        for obj in diverse:
            obj.pop('_score', None)

        # Planets prepended — always highest priority, include day-visible ones too
        planets = get_planet_visibility_skyfield(user, date_str) or []
        return planets + diverse

    except Exception as e:
        print(f"\n  Warning: visibility computation failed for {date_str}: {e}")
        return []

## Weekly loop

In [ ]:
def get_weekly_visibility(
    user       : UserProfile,
    observer   : Observer,
    targets    : list,
    valid_rows : list,
) -> list[dict]:
    """
    Computes visibility for all 7 nights. Observer and target list
    passed in — built once in Cell 6, never rebuilt.
    Gracefully handles per-night failures so one bad night doesn't
    crash the entire weekly computation.
    """
    today  = datetime.today()
    weekly = []

    for offset in range(7):
        date_str = (today + timedelta(days=offset)).strftime('%Y-%m-%d')
        print(f"Computing {date_str}...", end=" ")

        try:
            nightly = compute_nightly_visibility(
                observer, targets, valid_rows, date_str, user
            )
        except Exception as e:
            print(f"ERROR: {e}")
            nightly = []

        if nightly is None:
            nightly = []

        n_planets = sum(1 for o in nightly if o.get('is_solar_system'))
        n_dso     = sum(1 for o in nightly if not o.get('is_solar_system'))
        print(
            f"{len(nightly)} objects  "
            f"({n_planets} planets/moon + {n_dso} deep-sky)"
        )

        weekly.append({
            "date"                : date_str,
            "day_offset"          : offset,
            "visible_object_count": len(nightly),
            "objects"             : nightly,
        })

    return weekly

## Run and save

In [ ]:
weekly_visibility = get_weekly_visibility(
    plan_request.user, observer, targets, valid_rows
)

with open(f'{DATA_DIR}/weekly_visibility.json', 'w') as f:
    json.dump(weekly_visibility, f, indent=2, default=str)

print(f"\nSaved to {DATA_DIR}/weekly_visibility.json")
print("\n=== Visibility Summary ===")
for night in weekly_visibility:
    print(f"{night['date']}: {night['visible_object_count']} total objects")

Computing 2026-07-02... 21 objects  (6 planets/moon + 15 deep-sky)
Computing 2026-07-03... 21 objects  (6 planets/moon + 15 deep-sky)
Computing 2026-07-04... 21 objects  (6 planets/moon + 15 deep-sky)
Computing 2026-07-05... 21 objects  (6 planets/moon + 15 deep-sky)
Computing 2026-07-06... 21 objects  (6 planets/moon + 15 deep-sky)
Computing 2026-07-07... 21 objects  (6 planets/moon + 15 deep-sky)
Computing 2026-07-08... 21 objects  (6 planets/moon + 15 deep-sky)

Saved to /content/drive/MyDrive/AstroPlanner/data/weekly_visibility.json

=== Visibility Summary ===
2026-07-02: 21 total objects
2026-07-03: 21 total objects
2026-07-04: 21 total objects
2026-07-05: 21 total objects
2026-07-06: 21 total objects
2026-07-07: 21 total objects
2026-07-08: 21 total objects


 ## Spot check

In [ ]:
best = max(weekly_visibility, key=lambda n: n['visible_object_count'])
print(f"Best night: {best['date']}  ({best['visible_object_count']} objects)\n")

print("=== Solar System Bodies (true geometric times) ===")
for obj in best['objects']:
    if obj.get('is_solar_system'):
        period = obj.get('observable_period', 'night')
        period_label = {
            'night': '🌙 night only',
            'day'  : '☀️  day only',
            'both' : '🌓 day + night',
        }.get(period, period)
        print(
            f"  {obj['name']:8}  "
            f"peak={obj['peak_altitude_deg']}° at {obj['transit_time']}  "
            f"rises={obj['rise_time']}  sets={obj['set_time']}  "
            f"[{period_label}]"
        )

print("\n=== Top 15 Deep-Sky Objects ===")
dso = [o for o in best['objects'] if not o.get('is_solar_system')]
for obj in dso[:15]:
    moon_flag = "  ⚠ near moon" if obj.get('moon_warning') else ""
    dawn_flag = "  (transits after dawn)" if obj.get('transits_after_dawn') else ""
    print(
        f"  {obj['name']:10} {(obj['common_name'] or ''):25} "
        f"type={obj['type']:5} mag={str(obj['magnitude']):5} "
        f"peak={obj['peak_altitude_deg']}° at {obj['peak_time_local']}  "
        f"rise={obj['rise_time_local']}  set={obj['set_time_local']}"
        f"{moon_flag}{dawn_flag}"
    )

print("\n=== Object Type Distribution ===")
dist = defaultdict(int)
for o in best['objects']:
    dist[o['type']] += 1
for t, c in sorted(dist.items(), key=lambda x: -x[1]):
    print(f"  {t:10}: {c}")

print("\n=== Observable Period Breakdown ===")
for period, label in [('night', '🌙 night'), ('day', '☀️  day'), ('both', '🌓 both')]:
    count = sum(1 for o in best['objects'] if o.get('observable_period') == period)
    if count:
        print(f"  {label}: {count} objects")

Best night: 2026-07-02  (21 objects)

=== Solar System Bodies (true geometric times) ===
  Moon      peak=38.2° at 00:20  rises=18:59  sets=05:37  [🌙 night only]
  Jupiter   peak=77.2° at 11:20  rises=04:17  sets=18:17  [☀️  day only]
  Saturn    peak=60.2° at 04:00  rises=21:51  sets=10:12  [🌓 day + night]
  Mars      peak=77.3° at 07:10  rises=00:10  sets=14:09  [☀️  day only]
  Venus     peak=72.6° at 12:50  rises=06:03  sets=19:32  [☀️  day only]
  Mercury   peak=74.7° at 10:50  rises=04:02  sets=17:48  [☀️  day only]

=== Top 15 Deep-Sky Objects ===
  NGC6992    Eastern Veil,Network Nebula type=SNR   mag=7.0   peak=87.7° at 23:54  rise=19:19  set=up at dawn
  NGC6960    Veil Nebula,Filamentary Nebula,Western Veil type=SNR   mag=7.0   peak=87.4° at 23:54  rise=19:19  set=up at dawn
  NGC6995    Eastern Veil,Network Nebula type=SNR   mag=7.0   peak=87.2° at 23:54  rise=19:19  set=up at dawn
  NGC6720    Ring Nebula               type=PN    mag=8.8   peak=87.8° at 22:11  rise=up at d